# GraphNull on the competition graphs (CPU, competition data as input)

Runs the GraphNull library on every graph and embedding file of the Gemma 4 Developer Agent competition and checks that it reproduces the worlds used in the Map or Ritual study (manifest hash 5b88c0e967618d8b). It also builds the stratified null (degree and per-file neighbour counts kept) and reports its invariants. Everything is written to /tmp. Only counts and hashes are printed; no competition data is saved to the output. Apache 2.0.

In [ ]:
import os
os.makedirs('/tmp/gn', exist_ok=True)
os.chdir('/tmp/gn')
open('graphnull.py', 'w').write('# Copyright 2026 Yerramsetty Sai Venkata Suchita, Manjunath Mailey, Kanuma Jagan Mohan.\n# Licensed under the Apache License, Version 2.0.\n"""GraphNull: interface-matched null worlds for code-graph tools.\n\nA null world keeps everything an agent can see about a graph tool (tool names,\nnode set, output format, every node\'s degree) and scrambles only the content\nthat a structural claim depends on. If an agent does as well in the null world\nas in the real one, the gain cannot be credited to the structure.\n\nNull ladder, from weakest to strongest:\n  random      same node set and edge count, edges placed uniformly (degrees change)\n  degree      degree-preserving double-edge swaps (every node keeps its degree)\n  stratified  degree-preserving swaps that also keep, for every node, how many of\n              its neighbours sit in each group (for example each source file)\n  permuted    real graph, embedding vectors permuted across nodes\n\nOnly the standard library and numpy are needed. Every function is\ndeterministic given its seed. The harness file format is a JSON graph with\n"nodes" and "edges" (source, target, type) and an .npz embedding file keyed by\nnode id, as in the Gemma 4 Developer Agent competition.\n"""\nimport collections\nimport glob\nimport hashlib\nimport json\nimport os\nimport random\n\nimport numpy as np\n\n__version__ = "1.0.0"\nDEFAULT_SEED = 20261108\nSWAPS_PER_EDGE = 10\n\n\n# ---------------------------------------------------------------- seeds and io\ndef name_seed(name, seed=DEFAULT_SEED):\n    """Stable per-object seed derived from a global seed and a name."""\n    return int(hashlib.sha256(f"{seed}:{name}".encode()).hexdigest()[:12], 16)\n\n\ndef normalize(edges):\n    """Edge dicts or (u, v) tuples to a set of undirected pairs and a set of self loops."""\n    pairs, loops = set(), set()\n    for e in edges:\n        u, v = (e["source"], e["target"]) if isinstance(e, dict) else e\n        if u == v:\n            loops.add(u)\n        else:\n            pairs.add((u, v) if u < v else (v, u))\n    return pairs, loops\n\n\ndef degrees(pairs):\n    d = collections.Counter()\n    for u, v in pairs:\n        d[u] += 1\n        d[v] += 1\n    return d\n\n\ndef adjacency(pairs):\n    a = collections.defaultdict(set)\n    for u, v in pairs:\n        a[u].add(v)\n        a[v].add(u)\n    return a\n\n\ndef emit(graph, pairs, loops, edge_type="calls"):\n    """Write pairs back in the harness format: both directions, one edge type."""\n    edges = []\n    for u, v in sorted(pairs):\n        edges.append({"source": u, "target": v, "type": edge_type})\n        edges.append({"source": v, "target": u, "type": edge_type})\n    for u in sorted(loops):\n        edges.append({"source": u, "target": u, "type": edge_type})\n    out = {k: v for k, v in graph.items() if k not in ("edges", "links")}\n    out["edges"] = edges\n    return out\n\n\ndef write_json(obj, path):\n    blob = json.dumps(obj, sort_keys=True).encode()\n    with open(path, "wb") as f:\n        f.write(blob)\n    return hashlib.sha256(blob).hexdigest()\n\n\ndef file_hash(path):\n    with open(path, "rb") as f:\n        return hashlib.sha256(f.read()).hexdigest()\n\n\n# ---------------------------------------------------------------- null models\ndef rewire_degree(pairs, seed, swaps_per_edge=SWAPS_PER_EDGE, accept=None):\n    """Degree-preserving double-edge swaps.\n\n    Picks two edges (a, b) and (c, d) and replaces them with (a, d) and (c, b)\n    unless that creates a self loop or a duplicate edge. `accept(a, b, c, d)`\n    can veto a swap (used by the stratified null). Returns (new pair set, share of the\n    requested swaps that were completed).\n    """\n    rng = random.Random(seed)\n    E = sorted(pairs)\n    S = set(E)\n    m = len(E)\n    done, tries = 0, 0\n    while done < swaps_per_edge * m and tries < swaps_per_edge * m * 50:\n        tries += 1\n        i = int(rng.random() * m)\n        j = int(rng.random() * m)\n        if i == j:\n            continue\n        a, b = E[i]\n        c, d = E[j]\n        if rng.random() < 0.5:\n            c, d = d, c\n        if len({a, b, c, d}) < 4:\n            continue\n        if accept is not None and not accept(a, b, c, d):\n            continue\n        e1 = (a, d) if a < d else (d, a)\n        e2 = (c, b) if c < b else (b, c)\n        if e1 in S or e2 in S:\n            continue\n        S.discard(E[i])\n        S.discard(E[j])\n        S.add(e1)\n        S.add(e2)\n        E[i], E[j] = e1, e2\n        done += 1\n    return set(E), done / max(swaps_per_edge * m, 1)\n\n\ndef rewire_stratified(pairs, group_of, seed, swaps_per_edge=SWAPS_PER_EDGE):\n    """Degree-preserving swaps restricted to partners in the same group.\n\n    A swap (a, b), (c, d) -> (a, d), (c, b) is allowed only when b and d are in\n    the same group and a and c are in the same group. Then each of the four\n    endpoints gives up one neighbour and receives another from the same group,\n    so every node keeps both its degree and its count of neighbours per group. With group = source file this\n    keeps file locality and scrambles which function in the file is called.\n    """\n    return rewire_degree(pairs, seed, swaps_per_edge,\n                         accept=lambda a, b, c, d: group_of(b) == group_of(d) and group_of(a) == group_of(c))\n\n\ndef random_same_size(nodes, n_edges, seed):\n    """Uniform random simple graph on the same nodes with the same edge count."""\n    rng = random.Random(seed)\n    nodes = sorted(nodes)\n    out = set()\n    if len(nodes) < 2:\n        return out\n    cap = len(nodes) * (len(nodes) - 1) // 2\n    n_edges = min(n_edges, cap)\n    while len(out) < n_edges:\n        u, v = rng.sample(nodes, 2)\n        out.add((u, v) if u < v else (v, u))\n    return out\n\n\ndef permute_embeddings(keys, matrix, seed):\n    """Same keys, same vectors, assigned to different nodes."""\n    perm = np.random.default_rng(seed).permutation(len(keys))\n    return matrix[perm]\n\n\n# ---------------------------------------------------------------- checks\ndef group_profile(pairs, group_of):\n    """For every node, a Counter of neighbour groups."""\n    prof = collections.defaultdict(collections.Counter)\n    for u, v in pairs:\n        prof[u][group_of(v)] += 1\n        prof[v][group_of(u)] += 1\n    return prof\n\n\ndef check_null(real, null, group_of=None):\n    """Invariants of a null graph against the real one."""\n    out = {\n        "same_edge_count": len(real) == len(null),\n        "degree_preserved": degrees(real) == degrees(null),\n        "edges_changed": 1 - len(real & null) / max(len(real), 1),\n    }\n    if group_of is not None:\n        out["group_profile_preserved"] = group_profile(real, group_of) == group_profile(null, group_of)\n    return out\n\n\n# ---------------------------------------------------------------- harness worlds\ndef make_worlds(graph_dir, emb_dir, out_dir, seed=DEFAULT_SEED, swaps_per_edge=SWAPS_PER_EDGE,\n                stratified_group=None, verbose=True):\n    """Build harness-compatible worlds from a directory of graphs and embeddings.\n\n    W1 real graph (normalized), W2 degree-preserving rewire, W3 real graph with\n    permuted embeddings. If `stratified_group` is a function node_id -> group,\n    W2S (degree and group preserving) is also written. Graph and embedding files\n    keep their names, so a harness pointed at out_dir/W2 cannot tell the worlds\n    apart by interface. Returns (manifest dict, manifest sha256 prefix, report).\n    """\n    gfiles = {os.path.basename(p)[:-5]: p for p in glob.glob(os.path.join(graph_dir, "*.json"))}\n    efiles = {os.path.basename(p)[:-4]: p for p in glob.glob(os.path.join(emb_dir, "*.npz"))}\n    worlds = ["W1", "W2", "W3"] + (["W2S"] if stratified_group else [])\n    for w in worlds:\n        for sub in ("graphs", "embeddings"):\n            os.makedirs(os.path.join(out_dir, w, sub), exist_ok=True)\n    manifest = {"seed": seed, "files": {}}\n    report = collections.defaultdict(list)\n    for k, name in enumerate(sorted(gfiles)):\n        with open(gfiles[name]) as f:\n            G = json.load(f)\n        pairs, loops = normalize(G["edges"])\n        s = name_seed(name, seed)\n        rpairs, frac = rewire_degree(pairs, s, swaps_per_edge)\n        chk = check_null(pairs, rpairs)\n        report["swaps_completed"].append(frac)\n        report["degree_preserved"].append(chk["degree_preserved"])\n        report["edges_changed"].append(chk["edges_changed"])\n        g1 = os.path.join(out_dir, "W1", "graphs", name + ".json")\n        g2 = os.path.join(out_dir, "W2", "graphs", name + ".json")\n        g3 = os.path.join(out_dir, "W3", "graphs", name + ".json")\n        h1 = write_json(emit(G, pairs, loops), g1)\n        h2 = write_json(emit(G, rpairs, loops), g2)\n        if not os.path.lexists(g3):\n            os.symlink(os.path.abspath(g1), g3)\n        entry = {"W1_graph": h1, "W2_graph": h2}\n        if stratified_group:\n            spairs, _ = rewire_stratified(pairs, stratified_group, s, swaps_per_edge)\n            sc = check_null(pairs, spairs, stratified_group)\n            report["stratified_profile_preserved"].append(sc["group_profile_preserved"] and sc["degree_preserved"])\n            report["stratified_edges_changed"].append(sc["edges_changed"])\n            gs = os.path.join(out_dir, "W2S", "graphs", name + ".json")\n            entry["W2S_graph"] = write_json(emit(G, spairs, loops), gs)\n        if name in efiles:\n            src = efiles[name]\n            for w in ["W1", "W2"] + (["W2S"] if stratified_group else []):\n                dst = os.path.join(out_dir, w, "embeddings", name + ".npz")\n                if not os.path.lexists(dst):\n                    os.symlink(os.path.abspath(src), dst)\n            E = np.load(src)\n            keys = list(E.keys())\n            M = np.stack([E[x] for x in keys])\n            Mp = permute_embeddings(keys, M, s)\n            dst3 = os.path.join(out_dir, "W3", "embeddings", name + ".npz")\n            np.savez(dst3, **{x: Mp[i] for i, x in enumerate(keys)})\n            E3 = np.load(dst3)\n            report["embeddings_same_keys"].append(sorted(E3.keys()) == sorted(keys) and E3[keys[0]].dtype == M.dtype)\n            entry["emb"] = file_hash(src)\n            entry["W3_emb"] = file_hash(dst3)\n        if stratified_group:\n            manifest["files"][name] = entry\n        else:\n            manifest["files"][name] = {key: entry[key] for key in ("W1_graph", "W2_graph", "emb", "W3_emb") if key in entry}\n        if verbose and k % 25 == 0:\n            print(f"  graph {k + 1}/{len(gfiles)}")\n    blob = json.dumps(manifest, sort_keys=True).encode()\n    with open(os.path.join(out_dir, "manifest.json"), "wb") as f:\n        f.write(blob)\n    return manifest, hashlib.sha256(blob).hexdigest()[:16], dict(report)\n\n\ndef summarize(report):\n    """One line per invariant: count of passes and min / median of shares."""\n    lines = []\n    for key, vals in report.items():\n        if vals and isinstance(vals[0], bool):\n            lines.append(f"{key}: {sum(vals)} / {len(vals)}")\n        elif vals:\n            lines.append(f"{key}: min {min(vals):.3f} | median {float(np.median(vals)):.3f}")\n    return "\\n".join(lines)\n\n\nif __name__ == "__main__":\n    import argparse\n    ap = argparse.ArgumentParser(description="Build GraphNull worlds for a harness graph directory.")\n    ap.add_argument("graph_dir")\n    ap.add_argument("emb_dir")\n    ap.add_argument("out_dir")\n    ap.add_argument("--seed", type=int, default=DEFAULT_SEED)\n    ap.add_argument("--swaps-per-edge", type=int, default=SWAPS_PER_EDGE)\n    ap.add_argument("--stratify-by-file", action="store_true",\n                    help="also write W2S, grouping nodes by the module part of their id")\n    a = ap.parse_args()\n    grp = (lambda n: n.rsplit(".", 1)[0]) if a.stratify_by_file else None\n    _, h, rep = make_worlds(a.graph_dir, a.emb_dir, a.out_dir, a.seed, a.swaps_per_edge, grp)\n    print(summarize(rep))\n    print("manifest sha256:", h)\n')
print('wrote', 'graphnull.py', len(open('graphnull.py').read().splitlines()), 'lines')

In [ ]:
import glob, os, sys, time
sys.path.insert(0, "/tmp/gn")
import graphnull as gn
BASE = "/kaggle/input/competitions/gemma-4-developer-agent"
if not os.path.exists(f"{BASE}/tasks.jsonl"):
    hits = glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)
    assert hits, "Add the competition 'Gemma 4 Developer Agent' as Input first"
    BASE = os.path.dirname(hits[0])
t0 = time.time()
# 1. exactly the study's worlds (W1, W2, W3)
_, h, rep = gn.make_worlds(f"{BASE}/graphs", f"{BASE}/embeddings", "/tmp/gn/worlds", verbose=False)
print("=== GraphNull", gn.__version__, "on", len(rep["degree_preserved"]), "competition graphs ===")
print(gn.summarize(rep))
print("worlds manifest sha256:", h, "| study hash 5b88c0e967618d8b ->", "MATCH" if h == "5b88c0e967618d8b" else "DIFFERENT")
# 2. the stratified null, grouping nodes by source file (module part of the node id)
import json
def file_group_for(graph):
    g = {}
    for n in graph.get("nodes", []):
        nid = n.get("id") if isinstance(n, dict) else n
        path = (n.get("file") or n.get("file_path") or n.get("path")) if isinstance(n, dict) else None
        g[nid] = path or module_part(nid)
    return g
def module_part(nid):
    # module = leading lowercase segments of the id, before any CapWords class and the final name
    segs = nid.split(".")[:-1]
    out = []
    for s in segs:
        if s[:1].isupper():
            break
        out.append(s)
    return ".".join(out) or nid
n_file_field = 0
strat = {"profile_kept": 0, "graphs": 0, "changed": [], "completed": []}
for p in sorted(glob.glob(f"{BASE}/graphs/*.json")):
    G = json.load(open(p)); grp = file_group_for(G); pairs, _ = gn.normalize(G["edges"])
    gof = lambda x: grp.get(x, module_part(x))
    n_file_field += sum(1 for n in G.get("nodes", []) if isinstance(n, dict) and (n.get("file") or n.get("file_path") or n.get("path")))
    sp, frac = gn.rewire_stratified(pairs, gof, gn.name_seed(os.path.basename(p)[:-5]))
    c = gn.check_null(pairs, sp, gof)
    strat["graphs"] += 1; strat["profile_kept"] += c["degree_preserved"] and c["group_profile_preserved"]
    strat["changed"].append(c["edges_changed"]); strat["completed"].append(frac)
import numpy as np
print("\n=== stratified null (degree and per-file neighbour counts kept) ===")
print("nodes with a file field:", n_file_field, "(otherwise the module part of the node id is used)")
print("graphs:", strat["graphs"], "| degree and file profile kept:", strat["profile_kept"], "/", strat["graphs"])
print("edges changed median:", round(float(np.median(strat["changed"])), 3), "| swaps completed median:", round(float(np.median(strat["completed"])), 3))
print("files saved to notebook output:", len(os.listdir("/kaggle/working")))
print("seconds:", round(time.time() - t0))
print("GRAPHNULL COMPETITION CHECK DONE")